# Sensitivity study

## Experiment output folders

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dirs = [Path("output/L16B06")]

## Load test dataset

In [ ]:
import xarray as xr

## Open a netCDF file in a xarray dataset
fname = 'data/garachico2048.ens.nc'
ds    = xr.open_dataset(fname)
da    = ds['tephra_col_mass']
x_test_mean = da.mean(dim='ens').values

## Iterate over models

In [ ]:
import numpy as np
import torch
from modules.model import VariationalAutoencoder
from modules.dataset import MinMaxScale

nens     = 2048  ## Ensemble size
nsamples = 20    ## Number of ensemble generations

In [ ]:
metrics = []
for output_dir in output_dirs:
    ## Load weight parameters and some metadata
    fname = output_dir / 'model.pt'
    checkpoint = torch.load(fname)
    
    ## Recreate the model
    model = VariationalAutoencoder(checkpoint['LATENT_DIM'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    ## Normalization
    min_value = checkpoint['MINVAL']
    max_value = checkpoint['MAXVAL']
    transform = MinMaxScale(min_value, max_value)

    print(f"Working for {output_dir}")
    print(f"  - beta: {checkpoint['BETA']}")
    print(f"  - latent dim: {checkpoint['LATENT_DIM']}")
    for nsample in range(nsamples):
        ## Generate nens new samples
        z = torch.randn(nens, checkpoint['LATENT_DIM'])
        with torch.no_grad():
            new_sample = model.decode(z)
            x = transform.invert(new_sample).squeeze()

        ## Compute RMSE between means
        x_mean = x.numpy().mean(axis=0)
        rmse = np.sqrt(np.mean((x_mean - x_test_mean) ** 2))

        ## Store data
        metrics.append({
            'output_dir': str(output_dir),
            'nsample': nsample,
            'latent_dim': checkpoint['LATENT_DIM'],
            'beta': checkpoint['BETA'],
            'rmse': rmse,
        })
print("Done!")

## Save metrics to csv

In [ ]:
fname_out = "output_name.csv"
df = pd.DataFrame(metrics)
df.to_csv(fname_out)

## Load RMSE

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df1 = pd.read_csv("data/rmse_by_latent.csv")
df2 = pd.read_csv("data/rmse_by_beta.csv")

## Boxplots

In [ ]:
fig, axs = plt.subplots(
    nrows        = 2, 
    sharey       = True, 
    figsize      = (4,6), 
    tight_layout = True
)

df1.boxplot(by='latent_dim', column =['rmse'], ax=axs[0])
df2.boxplot(by='beta',       column =['rmse'], ax=axs[1])

axs[0].set(xlabel = 'Latent space dimension', 
           ylabel = r'RMSE [$g/m^2$]',
           title  = '(a)')
axs[1].set(xlabel = r'$\beta$', 
           ylabel = r'RMSE [$g/m^2$]',
           title  = '(b)')
plt.suptitle('')